# Delta Lake com Apache Spark (PySpark)

## TechStore — Sistema de Vendas

Este notebook demonstra as operações fundamentais do **Delta Lake** integrado ao **Apache Spark** usando o cenário de uma loja virtual chamada **TechStore**.

### O que será demonstrado:
- Configuração do ambiente PySpark + Delta Lake
- Geração de dados sintéticos com `Faker`
- **INSERT** — Escrita inicial de dados
- **UPDATE** — Atualização de registros
- **DELETE** — Remoção de registros
- **MERGE** — Upsert (INSERT + UPDATE combinados)
- **Time Travel** — Consulta de versões históricas
- **Histórico de transações** — Auditoria do Delta Log

### Modelo de Dados
```
clientes  ──┐
            ├─ pedidos ──┐
produtos  ──┘            └─ itens_pedido
```

---
## 1. Verificação do Ambiente

In [ ]:
import sys
import subprocess

print(f"Python: {sys.version}")

try:
    import pyspark
    print(f"PySpark: {pyspark.__version__}")
except ImportError:
    print("PySpark nao encontrado. Execute: uv sync")

try:
    import delta
    print(f"delta-spark: OK")
except ImportError:
    print("delta-spark nao encontrado. Execute: uv sync")

try:
    from faker import Faker
    print("Faker: OK")
except ImportError:
    print("Faker nao encontrado. Execute: uv sync")

---
## 2. Configuração da SparkSession com Delta Lake

In [ ]:
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    DoubleType, DateType
)
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

# Remover warehouse anterior para ambiente limpo
WAREHOUSE_PATH = "../warehouse/delta"
if os.path.exists(WAREHOUSE_PATH):
    shutil.rmtree(WAREHOUSE_PATH)
    print(f"Warehouse removido: {WAREHOUSE_PATH}")

# Configurar SparkSession com Delta Lake
builder = (
    SparkSession.builder
    .appName("TechStore-DeltaLake")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print(f"SparkSession iniciada com sucesso!")
print(f"Versao Spark : {spark.version}")
print(f"UI disponivel: http://localhost:4040")

---
## 3. Geração de Dados Sintéticos com Faker

In [ ]:
from faker import Faker
import random
from datetime import date, timedelta

fake = Faker('pt_BR')
Faker.seed(42)
random.seed(42)

# ─────────────────────────────────────────────
# CLIENTES
# ─────────────────────────────────────────────
ESTADOS = ['SC', 'SP', 'PR', 'RS', 'RJ', 'MG', 'BA', 'GO']
CIDADES = {
    'SC': ['Criciuma', 'Florianopolis', 'Blumenau', 'Joinville', 'Tubarao'],
    'SP': ['Sao Paulo', 'Campinas', 'Santos', 'Sorocaba'],
    'PR': ['Curitiba', 'Londrina', 'Maringa', 'Ponta Grossa'],
    'RS': ['Porto Alegre', 'Caxias do Sul', 'Pelotas'],
    'RJ': ['Rio de Janeiro', 'Niteroi', 'Petropolis'],
    'MG': ['Belo Horizonte', 'Uberlandia', 'Juiz de Fora'],
    'BA': ['Salvador', 'Feira de Santana', 'Vitoria da Conquista'],
    'GO': ['Goiania', 'Anapolis', 'Aparecida de Goiania'],
}

clientes_data = []
for i in range(1, 101):
    estado = random.choice(ESTADOS)
    cidade = random.choice(CIDADES[estado])
    clientes_data.append((
        i,
        fake.name(),
        f"cliente{i}@techstore.com",
        cidade,
        estado,
        fake.date_between(start_date=date(2022, 1, 1), end_date=date(2024, 12, 31)),
    ))

# ─────────────────────────────────────────────
# PRODUTOS
# ─────────────────────────────────────────────
CATEGORIAS_PRODUTOS = [
    ('Smartphone Samsung Galaxy S24',     'Smartphones',   3299.99,  45),
    ('iPhone 15 Pro',                     'Smartphones',   7999.99,  20),
    ('Notebook Dell Inspiron 15',         'Notebooks',     3899.99,  30),
    ('MacBook Air M2',                    'Notebooks',     9999.99,  15),
    ('Smart TV LG 55" 4K',                'TVs',           2799.99,  25),
    ('Smart TV Samsung 65" QLED',         'TVs',           4999.99,  12),
    ('Tablet iPad Air',                   'Tablets',       4499.99,  18),
    ('Tablet Samsung Galaxy Tab S9',      'Tablets',       3199.99,  22),
    ('Fone Bluetooth Sony WH-1000XM5',   'Acessorios',    1899.99,  60),
    ('Fone AirPods Pro 2',               'Acessorios',    2299.99,  35),
    ('Teclado Mecanico Keychron K2',      'Perifericos',    599.99,  50),
    ('Mouse Logitech MX Master 3',        'Perifericos',    699.99,  40),
    ('Monitor Dell 27" IPS',             'Monitores',     1899.99,  20),
    ('Monitor LG UltraWide 34"',         'Monitores',     2499.99,  15),
    ('SSD Externo Samsung T7 1TB',        'Armazenamento',  499.99,  80),
    ('HD Externo Seagate 2TB',            'Armazenamento',  349.99,  65),
    ('Roteador TP-Link AX3000',           'Redes',          499.99,  55),
    ('Webcam Logitech C920',              'Perifericos',    699.99,  45),
    ('Impressora HP LaserJet',            'Impressoras',   1299.99,  10),
    ('Caixa de Som JBL Charge 5',         'Acessorios',     999.99,  70),
    ('Console PlayStation 5',             'Games',         4299.99,   8),
    ('Console Xbox Series X',             'Games',         4199.99,  10),
    ('Controle DualSense PS5',            'Games',          499.99,  40),
    ('Nintendo Switch OLED',              'Games',         2899.99,  18),
    ('Carregador MagSafe 20W',            'Acessorios',     199.99, 100),
    ('Capa iPhone 15 Silicone',           'Acessorios',      99.99, 200),
    ('Cabo USB-C Anker 2m',              'Acessorios',      79.99, 150),
    ('Hub USB-C 7-em-1',                 'Perifericos',    299.99,  60),
    ('Suporte para Notebook',             'Acessorios',    149.99,  80),
    ('Webcam 4K Elgato Facecam',          'Perifericos',   1299.99,  20),
]

produtos_data = [
    (i+1, nome, cat, preco, estoque)
    for i, (nome, cat, preco, estoque) in enumerate(CATEGORIAS_PRODUTOS)
]

# ─────────────────────────────────────────────
# PEDIDOS
# ─────────────────────────────────────────────
STATUS_OPCOES = ['PENDENTE', 'PROCESSANDO', 'ENVIADO', 'ENTREGUE', 'CANCELADO']
pedidos_data = []
for i in range(1, 201):
    cliente_id = random.randint(1, 100)
    data_pedido = fake.date_between(start_date=date(2024, 1, 1), end_date=date(2024, 12, 31))
    status = random.choices(
        STATUS_OPCOES,
        weights=[10, 15, 20, 45, 10]
    )[0]
    valor_total = round(random.uniform(79.99, 15000.00), 2)
    pedidos_data.append((i, cliente_id, data_pedido, status, valor_total))

# ─────────────────────────────────────────────
# ITENS DE PEDIDO
# ─────────────────────────────────────────────
itens_data = []
item_id = 1
for pedido_id, _, _, _, _ in pedidos_data:
    n_itens = random.randint(1, 4)
    produtos_selecionados = random.sample(produtos_data, n_itens)
    for prod in produtos_selecionados:
        produto_id, _, _, preco, _ = prod
        qtd = random.randint(1, 3)
        itens_data.append((item_id, pedido_id, produto_id, qtd, preco))
        item_id += 1

print(f"Dados gerados:")
print(f"  Clientes     : {len(clientes_data):>5}")
print(f"  Produtos     : {len(produtos_data):>5}")
print(f"  Pedidos      : {len(pedidos_data):>5}")
print(f"  Itens pedido : {len(itens_data):>5}")

---
## 4. INSERT — Criação e Escrita das Tabelas Delta

In [ ]:
# ─── Schemas ────────────────────────────────
schema_clientes = StructType([
    StructField("cliente_id",   IntegerType(), False),
    StructField("nome",         StringType(),  False),
    StructField("email",        StringType(),  False),
    StructField("cidade",       StringType(),  True),
    StructField("estado",       StringType(),  True),
    StructField("data_cadastro",DateType(),     False),
])

schema_produtos = StructType([
    StructField("produto_id",IntegerType(), False),
    StructField("nome",      StringType(),  False),
    StructField("categoria", StringType(),  False),
    StructField("preco",     DoubleType(),  False),
    StructField("estoque",   IntegerType(), False),
])

schema_pedidos = StructType([
    StructField("pedido_id",   IntegerType(), False),
    StructField("cliente_id",  IntegerType(), False),
    StructField("data_pedido", DateType(),     False),
    StructField("status",      StringType(),  False),
    StructField("valor_total", DoubleType(),  False),
])

schema_itens = StructType([
    StructField("item_id",       IntegerType(), False),
    StructField("pedido_id",     IntegerType(), False),
    StructField("produto_id",    IntegerType(), False),
    StructField("quantidade",    IntegerType(), False),
    StructField("preco_unitario",DoubleType(),  False),
])

# ─── Criar DataFrames ────────────────────────
df_clientes = spark.createDataFrame(clientes_data, schema=schema_clientes)
df_produtos  = spark.createDataFrame(produtos_data,  schema=schema_produtos)
df_pedidos   = spark.createDataFrame(pedidos_data,   schema=schema_pedidos)
df_itens     = spark.createDataFrame(itens_data,     schema=schema_itens)

# ─── INSERT: Escrever como tabelas Delta ─────
def escrever_delta(df, nome_tabela, path):
    df.write.format("delta") \
        .mode("overwrite") \
        .save(f"{WAREHOUSE_PATH}/{path}")
    print(f"[OK] Tabela '{nome_tabela}' criada | {df.count()} registros")

escrever_delta(df_clientes, "clientes",     "clientes")
escrever_delta(df_produtos,  "produtos",     "produtos")
escrever_delta(df_pedidos,   "pedidos",      "pedidos")
escrever_delta(df_itens,     "itens_pedido", "itens_pedido")

print("\nEstrutura do warehouse:")
for root, dirs, files in os.walk(WAREHOUSE_PATH):
    level = root.replace(WAREHOUSE_PATH, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files[:3]:
            print(f"{indent}  {f}")

In [ ]:
# Leitura das tabelas Delta
def ler_delta(path):
    return spark.read.format("delta").load(f"{WAREHOUSE_PATH}/{path}")

print("=" * 50)
print("CLIENTES (primeiros 5)")
print("=" * 50)
ler_delta("clientes").show(5)

print("=" * 50)
print("PRODUTOS (primeiros 5)")
print("=" * 50)
ler_delta("produtos").show(5, truncate=False)

print("=" * 50)
print("PEDIDOS (primeiros 5)")
print("=" * 50)
ler_delta("pedidos").show(5)

---
## 5. UPDATE — Atualizando Registros

No Delta Lake, o UPDATE usa a API `DeltaTable.update()` ou SQL `UPDATE`.

In [ ]:
# Carregar tabela Delta
dt_clientes = DeltaTable.forPath(spark, f"{WAREHOUSE_PATH}/clientes")
dt_pedidos  = DeltaTable.forPath(spark, f"{WAREHOUSE_PATH}/pedidos")
dt_produtos = DeltaTable.forPath(spark, f"{WAREHOUSE_PATH}/produtos")

print("Estado ANTES do UPDATE:")
ler_delta("clientes").filter("cliente_id <= 5").show()

# ─── UPDATE 1: Atualizar cidade de um cliente ──────────────
print("\n[UPDATE 1] Atualizando cidade do cliente_id=1...")
dt_clientes.update(
    condition=F.col("cliente_id") == 1,
    set={"cidade": F.lit("Tubarao"), "estado": F.lit("SC")}
)

# ─── UPDATE 2: Atualizar preco de todos produtos de uma categoria ──
print("[UPDATE 2] Reajustando preco de Smartphones em 10%...")
dt_produtos.update(
    condition=F.col("categoria") == "Smartphones",
    set={"preco": F.round(F.col("preco") * 1.10, 2)}
)

# ─── UPDATE 3: Mudar status de pedidos antigos ──────────────
print("[UPDATE 3] Marcando pedidos ENVIADO->ENTREGUE (data < 2024-06-01)...")
dt_pedidos.update(
    condition=(F.col("status") == "ENVIADO") & (F.col("data_pedido") < F.lit("2024-06-01")),
    set={"status": F.lit("ENTREGUE")}
)

print("\nEstado APOS os UPDATEs:")
print("\nCliente 1 atualizado:")
ler_delta("clientes").filter("cliente_id = 1").show()

print("\nSmartphones com novo preco:")
ler_delta("produtos").filter("categoria = 'Smartphones'").show(truncate=False)

print("\nContagem de status nos pedidos:")
ler_delta("pedidos").groupBy("status").count().orderBy("status").show()

---
## 6. DELETE — Removendo Registros

In [ ]:
dt_pedidos  = DeltaTable.forPath(spark, f"{WAREHOUSE_PATH}/pedidos")
dt_itens    = DeltaTable.forPath(spark, f"{WAREHOUSE_PATH}/itens_pedido")

print("Contagem ANTES do DELETE:")
print(f"  Pedidos      : {ler_delta('pedidos').count()}")
print(f"  Itens pedido : {ler_delta('itens_pedido').count()}")

# ─── DELETE 1: Remover pedidos cancelados ──────────────────
print("\n[DELETE 1] Removendo pedidos CANCELADOS...")
pedidos_cancelados = ler_delta("pedidos").filter("status = 'CANCELADO'").select("pedido_id")
ids_cancelados = [r.pedido_id for r in pedidos_cancelados.collect()]
print(f"  Pedidos cancelados a remover: {len(ids_cancelados)}")

dt_pedidos.delete(condition=F.col("status") == F.lit("CANCELADO"))

# ─── DELETE 2: Remover itens dos pedidos cancelados ────────
print("[DELETE 2] Removendo itens dos pedidos cancelados...")
if ids_cancelados:
    dt_itens.delete(condition=F.col("pedido_id").isin(ids_cancelados))

# ─── DELETE 3: Remover cliente especifico (LGPD) ────────────
print("[DELETE 3] Removendo cliente_id=50 (solicitacao LGPD)...")
dt_clientes = DeltaTable.forPath(spark, f"{WAREHOUSE_PATH}/clientes")
dt_clientes.delete(condition=F.col("cliente_id") == 50)

print("\nContagem APOS o DELETE:")
print(f"  Pedidos      : {ler_delta('pedidos').count()}")
print(f"  Itens pedido : {ler_delta('itens_pedido').count()}")
print(f"  Clientes     : {ler_delta('clientes').count()} (era 100)")

---
## 7. MERGE — Upsert (INSERT + UPDATE combinados)

In [ ]:
# Simular carga de novos clientes via sistema externo:
# - Alguns ja existem (UPDATE)
# - Outros sao novos (INSERT)
novos_clientes_data = [
    (1,   "Alice Souza ATUALIZADA",  "alice@techstore.com",   "Criciuma",   "SC", date(2024, 1, 5)),  # update
    (3,   "Carlos Mendes NOVO EMAIL","carlos3@techstore.com", "Sao Paulo",  "SP", date(2022, 5, 20)), # update
    (101, "Novo Cliente 101",         "novo101@techstore.com", "Goiania",    "GO", date(2025, 1, 2)),  # insert
    (102, "Novo Cliente 102",         "novo102@techstore.com", "Salvador",   "BA", date(2025, 1, 3)),  # insert
    (103, "Novo Cliente 103",         "novo103@techstore.com", "Recife",     "PE", date(2025, 1, 4)),  # insert
]

df_novos = spark.createDataFrame(novos_clientes_data, schema=schema_clientes)

print("Dados para MERGE:")
df_novos.show()

dt_clientes = DeltaTable.forPath(spark, f"{WAREHOUSE_PATH}/clientes")

# MERGE
(
    dt_clientes.alias("destino")
    .merge(
        df_novos.alias("fonte"),
        "destino.cliente_id = fonte.cliente_id"
    )
    .whenMatchedUpdate(set={
        "nome":         "fonte.nome",
        "email":        "fonte.email",
        "cidade":       "fonte.cidade",
        "estado":       "fonte.estado",
    })
    .whenNotMatchedInsert(values={
        "cliente_id":   "fonte.cliente_id",
        "nome":         "fonte.nome",
        "email":        "fonte.email",
        "cidade":       "fonte.cidade",
        "estado":       "fonte.estado",
        "data_cadastro":"fonte.data_cadastro",
    })
    .execute()
)

print("\nResultado apos MERGE:")
print(f"Total de clientes: {ler_delta('clientes').count()} (era 99 apos delete, agora +3 novos)")

print("\nClientes atualizados (ids 1 e 3):")
ler_delta("clientes").filter("cliente_id IN (1, 3)").show()

print("\nNovos clientes inseridos (ids 101, 102, 103):")
ler_delta("clientes").filter("cliente_id >= 101").show()

---
## 8. Time Travel — Consultando Versões Históricas

In [ ]:
dt_clientes = DeltaTable.forPath(spark, f"{WAREHOUSE_PATH}/clientes")

print("=" * 60)
print("HISTORICO DE TRANSACOES DA TABELA clientes")
print("=" * 60)
dt_clientes.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

# ─── Time Travel por versao ──────────────────
print("\nVersao 0 (estado inicial - apos INSERT):")
df_v0 = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(f"{WAREHOUSE_PATH}/clientes")
print(f"  Total de registros na versao 0: {df_v0.count()}")
df_v0.filter("cliente_id = 1").show()

print("\nVersao atual (apos todos os UPDATE/DELETE/MERGE):")
df_atual = ler_delta("clientes")
print(f"  Total de registros na versao atual: {df_atual.count()}")
df_atual.filter("cliente_id = 1").show()

# ─── Time Travel via SQL ─────────────────────
print("\nTime Travel via SQL (VERSION AS OF 0):")
caminho = os.path.abspath(f"{WAREHOUSE_PATH}/clientes").replace("\\", "/")
spark.sql(f"""
    SELECT cliente_id, nome, cidade, estado
    FROM delta.`{caminho}`
    VERSION AS OF 0
    WHERE cliente_id <= 5
    ORDER BY cliente_id
""").show()

---
## 9. Análises com Spark SQL

In [ ]:
# Registrar tabelas como views temporarias para SQL
ler_delta("clientes").createOrReplaceTempView("clientes")
ler_delta("produtos").createOrReplaceTempView("produtos")
ler_delta("pedidos").createOrReplaceTempView("pedidos")
ler_delta("itens_pedido").createOrReplaceTempView("itens_pedido")

print("=" * 60)
print("TOP 10 CLIENTES POR VALOR DE PEDIDOS")
print("=" * 60)
spark.sql("""
    SELECT
        c.nome,
        c.cidade,
        c.estado,
        COUNT(p.pedido_id)         AS total_pedidos,
        ROUND(SUM(p.valor_total),2) AS receita_total
    FROM pedidos p
    JOIN clientes c ON p.cliente_id = c.cliente_id
    GROUP BY c.nome, c.cidade, c.estado
    ORDER BY receita_total DESC
    LIMIT 10
""").show(truncate=False)

print("=" * 60)
print("PRODUTOS MAIS VENDIDOS (por quantidade)")
print("=" * 60)
spark.sql("""
    SELECT
        pr.nome,
        pr.categoria,
        SUM(i.quantidade)           AS total_vendido,
        ROUND(SUM(i.quantidade * i.preco_unitario), 2) AS receita
    FROM itens_pedido i
    JOIN produtos pr ON i.produto_id = pr.produto_id
    GROUP BY pr.nome, pr.categoria
    ORDER BY total_vendido DESC
    LIMIT 10
""").show(truncate=False)

print("=" * 60)
print("RECEITA MENSAL DE 2024")
print("=" * 60)
spark.sql("""
    SELECT
        DATE_FORMAT(data_pedido, 'yyyy-MM') AS mes,
        COUNT(*) AS total_pedidos,
        ROUND(SUM(valor_total), 2) AS receita_mensal
    FROM pedidos
    GROUP BY mes
    ORDER BY mes
""").show()

---
## 10. Conclusão

### O que foi demonstrado neste notebook:

| Operação | Método utilizado | Resultado |
|----------|-----------------|----------|
| **INSERT** | `df.write.format("delta").save()` | 4 tabelas criadas |
| **UPDATE** | `DeltaTable.update()` | Cidades, preços e status atualizados |
| **DELETE** | `DeltaTable.delete()` | Pedidos cancelados e cliente removido |
| **MERGE** | `DeltaTable.merge()` | Upsert: 2 atualizados + 3 inseridos |
| **Time Travel** | `versionAsOf` / `VERSION AS OF` | Consulta de versão histórica |
| **Histórico** | `DeltaTable.history()` | Auditoria completa de operações |

### Principais vantagens do Delta Lake:
- **ACID completo** em cima de arquivos Parquet simples
- **Time Travel** para auditoria e reprodutibilidade
- **Merge/Upsert** eficiente sem reescrever toda a tabela
- **Schema Evolution** sem quebrar pipelines existentes
- **Integração nativa com Spark** — mesma API de DataFrame

In [ ]:
# Encerrar SparkSession
spark.stop()
print("SparkSession encerrada.")